This notebook performs inference for a specific model on all unlabeled images in
the data/ folder.


In [ ]:
from pathlib import Path

import torch
from torchvision.io import read_image
from torchvision.utils import draw_bounding_boxes

from colony_segmentation.dataset_transforms import get_transforms
from colony_segmentation.model import get_model_instance_segmentation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


eval_transform = get_transforms(is_train=False)


def draw_boxes_on_image(image: torch.Tensor, boxes: torch.Tensor):
    image = draw_bounding_boxes(
        image,
        boxes,
        colors="#ff1111",
        width=8,
        font_size=52,
        font=str(Path("./assets/zed-mono-bold.ttf")),
    )

    return image


model = get_model_instance_segmentation(2).to(device)
model.load_state_dict(
    torch.load(
        "colony_segmentation/artifacts/2024-06-05_16-12/final.pth", map_location=device
    )
)


def predict(image: torch.Tensor):
    model.eval()
    with torch.no_grad():
        x = eval_transform(image)
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    return image, pred["boxes"].long(), pred["scores"]

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Specify the path
path = Path("data/unlabeled")

# Get all jpg files
jpg_files = sorted(list(path.glob("*.jpg")))

# Loop through the files and plot them
for jpg_file in tqdm(jpg_files):
    image = read_image(jpg_file)

    _, pred_boxes, __ = predict(image)
    pred_image = draw_boxes_on_image(image, pred_boxes)

    plt.figure(figsize=(10, 10))
    plt.imshow(pred_image.permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.show()